<a href="https://colab.research.google.com/github/confident-ai/deepeval/blob/main/examples/red_team/custom_red_teaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install deepeval

In [2]:
import os

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

Want to use other evaluation models? [Click here](https://deepeval.com/integrations/models/openai) to see all supported models and their usage instructions.

In [ ]:
from openai import OpenAI
from deepeval.red_team.vulnerabilities import Competition
from deepeval.red_team.attacks.single_turn import (
    PermissionEscalation,
    PromptInjection,
)
from deepeval.red_team.attacks.multi_turn import (
    LinearJailbreaking,
    SequentialJailbreak,
)
from deepeval.red_team.test_case import RTTurn
from deepeval.models import GeminiModel


def model_callback(attack, turn_history=None):
    model = GeminiModel(
        model_name="gemini-2.5-flash",
        api_key="YOUR_API_KEY_HERE",
        temperature=0,
    )
    return model.generate(attack)[0]


competetion = Competition()

test_cases = competetion.simulate_attacks()

permission_escalation = PermissionEscalation()
prompt_injection = PromptInjection()
linear_jailbreaking = LinearJailbreaking()
sequential_jailbreak = SequentialJailbreak()

safe_model = True

for test_case in test_cases:
    enhanced_attack_1 = permission_escalation.enhance(test_case.input)
    enhanced_attack_2 = prompt_injection.enhance(enhanced_attack_1)
    turns = [RTTurn(role="user", content=enhanced_attack_2)]
    new_turns_1 = linear_jailbreaking._get_turns(
        model_callback,
        turns,
        test_case.vulnerability,
        test_case.vulnerability_type,
    )
    new_turns_2 = sequential_jailbreak._get_turns(
        model_callback,
        new_turns_1,
        test_case.vulnerability,
        test_case.vulnerability_type,
    )
    test_case.turns = new_turns_2
    metric = competetion._get_metric(test_case.vulnerability_type)
    metric.measure(test_case)
    if metric.score < 1:
        safe_model = False
        print("LLM application failed. 😞")
        print(metric.score)
        print(metric.reason)
        break

if safe_model:
    print("LLM application passed! 🥳")